# 1. Context

This notebook pulls relevant metadata to download PDFs in languages supported in Project EKA 

In [1]:
from datasets import get_dataset_split_names, get_dataset_config_names, load_dataset
from pathlib import Path

# 2. EKA Supported Languages

In [2]:
eka_lang_codes = ["hin", "mar", "san", "kok", "nep" ,"mai", "guj","ben","asm",
                  "mni","bpy","tel","kan","tam","mal","urd","kas","snd","pan",
                  "ori","eng","zho","pus","bal","mya","doi","sat","brx"]

In [3]:
import os

In [4]:
os.chdir("/Users/niteshkumarsharma/Desktop/Folder/AI/indic_document_extraction/")

In [5]:
import pandas as pd

In [6]:
len(eka_lang_codes)

28

# 3. FinePDF Dataset

In [7]:
available_subsets = get_dataset_config_names("HuggingFaceFW/finepdfs")

In [8]:
## checking for eka language code in availabe subset
eka_lang_code_ss = []
for lang_eka in eka_lang_codes:
    for fine_pdf_lg_script in available_subsets:
        if lang_eka in fine_pdf_lg_script.split("_")[0]:
            eka_lang_code_ss.append(fine_pdf_lg_script)

In [9]:
import tqdm

In [10]:
len(eka_lang_code_ss)

42

In [11]:
_PATH_DS_FOLDER_ = Path("/Users/niteshkumarsharma/Desktop/Folder/AI/indic_document_extraction/finePDF_eka_metadata")

In [12]:
split_downloaded = [fp for fp in _PATH_DS_FOLDER_.glob("*.csv")]

In [13]:
df_concat = pd.concat([pd.read_csv(x) for x in split_downloaded])

In [14]:
df_concat.loc[df_concat['language'] == 'hin_Deva']

,language,split,ht_model_used,cc_dump_id,cc_s3_path,offset,pdf_url,trucated
0,hin_Deva,train,rolmOCR,CC-MAIN-2023-40,s3://commoncrawl/cc-index/table/cc-main/warc/c...,1016055073,https://www.nios.ac.in/media/documents/srsec30...,False
1,hin_Deva,train,docling,CC-MAIN-2024-30,s3://commoncrawl/cc-index/table/cc-main/warc/c...,277151140,https://jandaschool.com/pdf/SocietyCertificate...,True
2,hin_Deva,train,docling,CC-MAIN-2022-33,s3://commoncrawl/cc-index/table/cc-main/warc/c...,233284684,https://edustud.nic.in//edu/PracticePaper_2020...,False
3,hin_Deva,train,docling,CC-MAIN-2021-49,s3://commoncrawl/cc-index/table/cc-main/warc/c...,1116962722,https://www.rsmssb.rajasthan.gov.in/Static/fil...,False
4,hin_Deva,train,docling,CC-MAIN-2019-13,s3://commoncrawl/crawl-data/CC-MAIN-2019-13/se...,394512849,https://agamatrix.co.uk/wp-content/uploads/201...,False
...,...,...,...,...,...,...,...,...
850565,hin_Deva,test,docling,CC-MAIN-2021-39,s3://commoncrawl/cc-index/table/cc-main/warc/c...,222450520,https://cmdpgcollege.ac.in/Uploads/MA%20ECONOM...,False
850566,hin_Deva,test,docling,CC-MAIN-2023-06,s3://commoncrawl/cc-index/table/cc-main/warc/c...,696890024,https://www.ccsuniversity.ac.in/ccsum/34-convo...,False
850567,hin_Deva,test,docling,CC-MAIN-2023-40,s3://commoncrawl/cc-index/table/cc-main/warc/c...,275497495,https://finance.cg.gov.in/Transfer%20List/2022...,False
850568,hin_Deva,test,docling,CC-MAIN-2019-39,s3://commoncrawl/crawl-data/CC-MAIN-2019-39/se...,141003874,http://rdd.bih.nic.in/Circulars/432515.pdf,False


In [ ]:
2086593/2116183

In [ ]:
df_concat['ht_model_used'].value_counts()

In [ ]:
df_concat['language'].value_counts()

In [ ]:
dataset = load_dataset("HuggingFaceFW/finepdfs",name=split_download_req[0], split="train", streaming=True)

In [ ]:
dataset

In [ ]:
for eka_lang_script in tqdm.tqdm(split_download_req):
    metadata_list = []
    splits = get_dataset_split_names("HuggingFaceFW/finepdfs",config_name=eka_lang_script)
    for split in splits:
        dataset = load_dataset("HuggingFaceFW/finepdfs",name=eka_lang_script, split=split, streaming=True)
        for sample in dataset:
            try:
                metadata_list.append({"language": eka_lang_script,
                                      "split": split,
                                      "ht_model_used": sample.get("extractor"),
                                      "cc_dump_id": sample.get("dump"),
                                      "cc_s3_path": sample.get("file_path"),
                                      "offset": sample.get("offset"),
                                      "pdf_url": sample.get("url"),
                                      "trucated": sample.get("is_truncated"),
                                      })
            except Exception as E:
                print(E)
                continue
    df_req = pd.DataFrame(metadata_list)
    df_req.to_csv(f"finePDF_eka_metadata/{eka_lang_script}.csv", index=False)